# Cleaning Validation — Re-run EDA checks on the *cleaned* data

**Goal:** prove the cleaning actually worked, with **before/after evidence** an
auditor can read at a glance. We re-run the §2 EDA checks (#11–#14) on the output
of the real cleaning pipeline and confirm:

1. exact duplicates → **0**,
2. constant / corrupt images → **0**,
3. no malformed rows,
4. class distribution shape **preserved** (no over-cleaning skew).

We also run the **`stages.cleaning` toggle ON vs OFF** to quantify exactly how many
rows cleaning removes — the ablation the Architecture is built for.

This closes the data loop: §2 (problems) → §3 (strategies) → **§4 (what we did +
measured result)**. The numbers printed here populate `data.md §4`.

## 0. Setup

In [ ]:
import sys
import copy
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent  # notebooks/ → project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.logging import setup_logging
from src.emotion_detector.data.fer2013 import Fer2013Fetcher
from src.emotion_detector.data.cleaning import clean_dataset
from src.emotion_detector.data.imbalance import resolve_imbalance

cfg = load_config(ROOT / "config.yaml")
for key in cfg["paths"]:
    cfg["paths"][key] = str(ROOT / cfg["paths"][key])
setup_logging(cfg)

EDA_DIR = Path(cfg["paths"]["results_dir"]) / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

EMOTION_LABELS = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad",     5: "Surprise", 6: "Neutral",
}
print("cleaning config:", cfg["cleaning"])

In [ ]:
# --- metrics used for before/after comparison (same maths as notebooks 01/03/04) ---
def duplicate_count(images):
    """Number of exact-duplicate rows (MD5 of pixel bytes)."""
    h = [hashlib.md5(images[i].tobytes()).hexdigest() for i in range(len(images))]
    return int(pd.Series(h).duplicated().sum())


def constant_count(images):
    """Number of constant (std == 0) images."""
    return int((images.reshape(len(images), -1).std(axis=1) == 0).sum())


def malformed_count(images, size=48):
    """Number of images not shaped (size, size)."""
    return int(sum(im.shape != (size, size) for im in images))


def class_distribution(labels):
    """{emotion_name: count} in label order."""
    vc = pd.Series(labels).value_counts().sort_index()
    return {EMOTION_LABELS[int(k)]: int(v) for k, v in vc.items()}

## 1. Load the raw training split (before cleaning)

We validate on the **Training** split — the only split cleaning modifies
(val/test are never resampled or reweighted, CONTRIBUTING §8).

In [ ]:
fetcher = Fer2013Fetcher(cfg)
X_raw, y_raw = fetcher.fetch("Training")
print(f"Raw training: {X_raw.shape}, {len(y_raw):,} labels")

## 2. Run the real cleaning pipeline

`clean_dataset` applies exactly the config-selected cleaners (`DuplicateRemover`,
`CorruptImageRemover`) — the same code the training pipeline uses.

In [ ]:
X_clean, y_clean = clean_dataset(cfg, X_raw, y_raw)

summary = pd.DataFrame({
    "before": [len(X_raw), duplicate_count(X_raw), constant_count(X_raw), malformed_count(X_raw)],
    "after":  [len(X_clean), duplicate_count(X_clean), constant_count(X_clean), malformed_count(X_clean)],
}, index=["rows", "exact_duplicates", "constant_images", "malformed_rows"])
summary["removed"] = summary["before"] - summary["after"]
summary

## 3. Assert the cleaning invariants hold

The whole point of re-measuring: turn "we ran a cleaner" into a **proof**.

In [ ]:
assert duplicate_count(X_clean) == 0, "duplicates remain!"
assert constant_count(X_clean) == 0, "constant images remain!"
assert malformed_count(X_clean) == 0, "malformed rows remain!"
assert len(X_clean) == len(y_clean), "images/labels desynced!"
print("✓ duplicates = 0")
print("✓ constant images = 0")
print("✓ malformed rows = 0")
print("✓ images and labels aligned")
print(f"\nRemoved {len(X_raw) - len(X_clean):,} rows "
      f"({(len(X_raw) - len(X_clean)) / len(X_raw):.2%} of the training split).")

## 4. Class distribution before vs after (over-cleaning guard)

Removing duplicates and blanks could *skew* the class balance. We plot before vs
after proportions side-by-side — they should be **almost identical**, proving we
didn't distort the distribution.

In [ ]:
dist_before = class_distribution(y_raw)
dist_after = class_distribution(y_clean)
names = list(EMOTION_LABELS.values())

dist_tbl = pd.DataFrame({
    "before": [dist_before.get(n, 0) for n in names],
    "after":  [dist_after.get(n, 0) for n in names],
}, index=names)
dist_tbl["removed"] = dist_tbl["before"] - dist_tbl["after"]
dist_tbl["before_%"] = (dist_tbl["before"] / dist_tbl["before"].sum() * 100).round(2)
dist_tbl["after_%"] = (dist_tbl["after"] / dist_tbl["after"].sum() * 100).round(2)
display(dist_tbl)

x = np.arange(len(names))
w = 0.4
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, dist_tbl["before_%"], w, label="before", color="lightsteelblue")
ax.bar(x + w/2, dist_tbl["after_%"],  w, label="after",  color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("% of training split")
ax.set_title("Class distribution before vs after cleaning (proportions preserved)")
ax.legend()
plt.tight_layout()
fig.savefig(EDA_DIR / "clean_class_dist_before_after.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'clean_class_dist_before_after.png'}")
plt.show()

## 5. Ablation — `stages.cleaning` ON vs OFF

Flip the stage toggle and re-run: OFF must return the data untouched, ON removes
the bad rows. The delta is cleaning's exact contribution to the row count.

In [ ]:
cfg_off = copy.deepcopy(cfg)
cfg_off["stages"]["cleaning"] = False
X_off, y_off = clean_dataset(cfg_off, X_raw, y_raw)

ablation = pd.DataFrame({
    "rows": [len(X_off), len(X_clean)],
    "duplicates": [duplicate_count(X_off), duplicate_count(X_clean)],
    "constant": [constant_count(X_off), constant_count(X_clean)],
}, index=["stage OFF", "stage ON"])
display(ablation)

assert len(X_off) == len(X_raw), "stage OFF must not change the data!"
print("✓ stage OFF returns data unchanged")
print(f"✓ stage ON removes {len(X_off) - len(X_clean):,} rows")

## 6. Imbalance handling on the cleaned training set

`resolve_imbalance` applies the configured remedy to the **cleaned** training
split. With the default `class_weight`, the row count is unchanged and a weight
dict is produced for `model.fit`; `oversample`/`undersample` would instead
rebalance the counts (shown for reference).

In [ ]:
X_bal, y_bal, class_weight = resolve_imbalance(cfg, X_clean, y_clean)

print(f"Active strategy: {cfg['cleaning']['imbalance_strategy']}")
print(f"Rows after imbalance step: {len(X_bal):,} (was {len(X_clean):,})")
if class_weight is not None:
    print("\nClass weights (→ model.fit(class_weight=...)):")
    for c, w in sorted(class_weight.items()):
        print(f"  {EMOTION_LABELS[c]:<9} {w:.3f}")

# For reference: what oversample / undersample would produce on this data.
for strat in ("oversample", "undersample"):
    cfg_s = copy.deepcopy(cfg)
    cfg_s["cleaning"]["imbalance_strategy"] = strat
    _, y_s, _ = resolve_imbalance(cfg_s, X_clean, y_clean)
    print(f"\n[{strat}] rows: {len(y_s):,}, per-class: {class_distribution(y_s)}")

## 7. Findings for `data.md` §4

Fill the numbers below from this run (they are printed above), then copy into
`data.md §4`.

| Step | Problem | `config.yaml` | Before | After |
|---|---|---|---|---|
| Remove exact duplicates | P2.2 / P2.3 | `remove_duplicates: true` | _dup=?_ | 0 |
| Drop constant images | P2.5 | `drop_constant_images: true` | _const=?_ | 0 |
| Class weighting | P2.1 | `imbalance_strategy: class_weight` | — | weights emitted |
| Non-face filter | P2.6 | `drop_non_faces: false` | (off) | — |

- Training rows: _before_ → _after_ (_%_ removed).
- Class distribution shape preserved? (compare `before_%` vs `after_%` in §4).
- Stage ON vs OFF delta = _N_ rows (cleaning's measured contribution).